Setup

In [1]:
import numpy as np
import scanpy as sc
import pandas as pd
import liana as li
import gseapy as gp
from gseapy import Msigdb

import recon
import recon.data
import recon.explore

Load data

In [ ]:
rna = sc.read_h5ad("./data/perturbation_tuto/rna.h5ad", backed="r")
rna = rna[:5000, :1000].copy()

ValueError: To copy an AnnData object in backed mode, pass a filename: `.copy(filename='myfilename.h5ad')`. To load the object into memory, use `.to_memory()`.

In [4]:
df = rna.to_df()
print(df.head())

                     Xkr4  Gm1992  Gm37381  Rp1  Sox17  Gm37323  Mrpl15  \
ACTTCGCCACCTTCCA-21   0.0     0.0      0.0  0.0    0.0      0.0     0.0   
CTGAGCGGTGTCTTAG-09   0.0     0.0      0.0  0.0    0.0      0.0     1.0   
AGAAATGGTTCTTGCC-08   0.0     0.0      0.0  0.0    0.0      0.0     0.0   
ACCCTCAAGAGGTCGT-17   0.0     0.0      0.0  0.0    0.0      0.0     0.0   
GTGGGAATCACTACGA-28   0.0     0.0      0.0  0.0    0.0      0.0     0.0   

                     Lypla1  Gm37988  Tcea1  ...  Lhx3  Gm27196  Qsox2  \
ACTTCGCCACCTTCCA-21     0.0      0.0    0.0  ...   0.0      0.0    0.0   
CTGAGCGGTGTCTTAG-09     0.0      0.0    2.0  ...   0.0      0.0    0.0   
AGAAATGGTTCTTGCC-08     0.0      0.0    0.0  ...   0.0      0.0    0.0   
ACCCTCAAGAGGTCGT-17     0.0      0.0    0.0  ...   0.0      0.0    0.0   
GTGGGAATCACTACGA-28     0.0      0.0    0.0  ...   0.0      0.0    0.0   

                     Ccdc187  Gpsm1  Dnlz  Card9  Snapc4  Gm13563  Sdccag3  
ACTTCGCCACCTTCCA-21      0.

In [3]:
rna.obs["celltype"].unique().tolist()

['B_cell',
 'ILC',
 'Macrophage',
 'MigDC',
 'Monocyte',
 'NK_cell',
 'Neutrophil',
 'T_cell_CD4',
 'T_cell_CD8',
 'T_cell_gd',
 'Treg',
 'cDC1',
 'cDC2',
 'eTAC',
 'pDC']

## Build the Multilayer Network

### Gene regulatory network

In [4]:
# Load pre-computed GRN (or generate with ReCoN - see Tutorial 4)
grn_path = "./data/perturbation_tuto/grn.csv"
grn = pd.read_csv(grn_path)
grn = grn.sort_values(by="weight", ascending=False)[:500_000]
grn["source"] = grn["source"].str.capitalize()
grn["source"] = grn["source"] + '_TF'
grn["target"] = grn["target"].str.capitalize()
grn.head(3)

,Unnamed: 0,target,source,weight
0,0,Pax5,Mbd1_TF,0.000095
2,2,Pax5,Smad5_TF,0.000092
1,1,Pax5,Smad1_TF,0.000092


### Compute Cell-Cell Communication

In [5]:
li.method.cellphonedb(rna, 
            # NOTE by default the resource uses HUMAN gene symbols
            resource_name="mouseconsensus",
            expr_prop=0.00,
            use_raw=False,
            groupby="celltype",
            verbose=True, key_added='cpdb_res')

Using resource `mouseconsensus`.
Using `.X`!
/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
901 features of mat are empty, they will be removed.
Make sure that normalized counts are passed!
/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/liana/method/_pipe_utils/_pre.py:146: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/liana/method/_pipe_utils/_pre.py:149: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
0.94 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 1296 samples and 15 features


100%|██████████| 1000/1000 [00:02<00:00, 484.75it/s]


In [6]:
ccc_network = rna.uns["cpdb_res"].copy()
ccc_network = ccc_network[["ligand", "receptor", "lr_means", "source", "target"]]
ccc_network = ccc_network.rename(columns={
    "lr_means": "weight",
    "source": "celltype_source",
    "target": "celltype_target",
    "ligand": "source",
    "receptor": "target"
})
ccc_network = ccc_network[ccc_network['weight'] != 0]

Load Receptor-Gene Links

In [7]:
# Load receptor-gene links from NicheNet prior knowledge
receptor_genes = recon.data.load_data.load_receptor_genes("mouse_receptor_gene_from_NichenetPKN")
# for human, use "human_receptor_gene_from_NichenetPKN"

# Filter to genes present in our GRN
genes = np.unique(grn['source'].tolist() + grn['target'].tolist())
receptor_genes = receptor_genes[receptor_genes['target'].isin(genes)]
receptor_genes.head()

,source,target,weight
2,A1bg,Abca1,0.005156
3,A1bg,Abcb1a,0.005877
4,A1bg,Abcb1b,0.005877
7,A1bg,Acsl1,0.005915
8,A1bg,Adk,0.005092


## Define Seed Genes

Seeds are the genes of interest that define your biological question. ReCoN will explore the network to find regulators (upstream) or targets (downstream) of these seeds.

Example: We use the TNF-α signaling via NF-κB hallmark gene set, hypothetically activated in macrophages.

In [8]:
import gseapy as gp
from gseapy import Msigdb 

# we can use gseapy to download the hallmarks from MSigDB
msig = Msigdb()
hallmarks = msig.get_gmt(category='mh.all', dbver="2024.1.Mm")
print(f"Available hallmarks: {list(hallmarks.keys())[:5]}...")

# Select TNF-α signaling pathway and filter to genes in our network
gene_seeds = [gene for gene in hallmarks['HALLMARK_TNFA_SIGNALING_VIA_NFKB'] if gene in genes]
print(f"\nFiltered seeds: {len(gene_seeds)} genes from hallmark")

# Create seed dictionary with equal weights (all genes equally important)
gene_seeds = {seed: 1 for seed in gene_seeds}

Available hallmarks: ['HALLMARK_ADIPOGENESIS', 'HALLMARK_ALLOGRAFT_REJECTION', 'HALLMARK_ANDROGEN_RESPONSE', 'HALLMARK_ANGIOGENESIS', 'HALLMARK_APICAL_JUNCTION']...

Filtered seeds: 157 genes from hallmark


Now we need to assign seeds to a specific cell type. Since we’re investigating TNF-α signaling activated in macrophages, we add the ::Macrophage suffix to each gene:

In [9]:
# Format seeds for ReCoN: gene::celltype
# This tells ReCoN that these genes are activated specifically in Macrophages
seeds = {f"{gene}::Macrophage": score for gene, score in gene_seeds.items() if score > 0}

print(f"Example seeds: {dict(list(seeds.items())[:3])}")

Example seeds: {'Abca1::Macrophage': 1, 'Atf3::Macrophage': 1, 'Atp2b1::Macrophage': 1}


## Assemble the Multicellular Network

Now we build the Multicell object that integrates GRNs, receptor-gene links, and cell-cell communication across all cell types.

In [10]:
# Network parameters
cell_communication_graph_directed = True
cell_communication_graph_weighted = True
grn_graph_directed = True
grn_graph_weighted = True

# RWR parameters
restart_proba = 0.6  # Higher = stay closer to seeds
ccc_proba = 0.5      # Balance GRN vs CCC exploration

In [11]:
# Define cell types to include
celltypes = ["B_cell", "pDC", "Macrophage", "NK_cell", "T_cell_CD4", "T_cell_CD8"]

# Build the Multicell network
generic_multicell = recon.explore.Multicell(
    celltypes={
        celltype: recon.explore.Celltype(
            grn_graph=grn,
            receptor_grn_bipartite=receptor_genes,
            celltype_name=celltype,
            receptor_graph_directed=True,
            receptor_graph_weighted=False,
            grn_graph_directed=grn_graph_directed,
            grn_graph_weighted=grn_graph_weighted,
            receptor_grn_bipartite_graph_directed=True,
            receptor_grn_bipartite_graph_weighted=True,
            seeds=seeds
        )
        for celltype in celltypes
    },
    cell_communication_graph=ccc_network[
        ccc_network["celltype_source"].isin(celltypes) & 
        ccc_network["celltype_target"].isin(celltypes)
    ],
    cell_communication_graph_directed=cell_communication_graph_directed,
    cell_communication_graph_weighted=cell_communication_graph_weighted,
    bipartite_grn_cell_communication_directed=False,
    bipartite_grn_cell_communication_weighted=False,
    bipartite_cell_communication_receptor_directed=False,
    bipartite_cell_communication_receptor_weighted=False,
    seeds=seeds,
)

/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/recon/explore/recon.py:122: UserWarning: 
                No receptor_graph provided,
                an empty receptor graph will be created.
                
/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/recon/explore/recon.py:387: UserWarning: The celltypes dictionary was converted toa list of Celltype objects.
The keys of the dictionary will be the celltype names.


## Configure Exploration Direction

ReCoN allows you to explore the network in different directions:

- upstream: Find regulators that may have activated your seed genes
- downstream: Find targets that may be affected by your seed genes

And with different strategies:

- intracell: Explore only within the GRN (no cell-cell communication)
- intercell: Explore both GRN and cell-cell communication

In [ ]:
# Set lambda transition matrix for upstream intercellular exploration
generic_multicell.lamb = recon.explore.set_lambda(
    generic_multicell,
    direction="upstream",   # Look for upstream regulators
    strategy="intercell",   # Include cell-cell communication
)

## Run Random Walk with Restart

In [ ]:
# Create multiXrank object
multilayer = generic_multicell.Multixrank(
    restart_proba=restart_proba
)

# Run random walk with restart
results = multilayer.random_walk_rank()

In [ ]:
# Format results as gene profiles per cell type
cell_type_profiles = recon.explore.format_multicell_results(
    multicell_multixrank_results=results,
    celltypes=celltypes,
    keep_layers="gene"
)

cell_type_profiles.head()

## Interpret Results

The results show RWR scores for each gene in each cell type. Higher scores indicate genes more strongly connected to your seeds through the multilayer network.


You can use these cell-type-specific gene profiles for:

- Gene set enrichment analysis to identify activated pathways per cell type
- Visualization with Sankey diagrams (see Tutorial 3)
- Comparison between conditions or treatments

In [ ]:
cell_type_profiles

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check if 'gene' is a column or the index
if 'gene' not in cell_type_profiles.columns:
    cell_type_profiles = cell_type_profiles.reset_index().rename(columns={'index': 'gene'})

# Get top 10 genes per cell type
n_top = 10
top_genes_per_celltype = {}
for ct in celltypes:
    if ct in cell_type_profiles.columns:
        top_genes_per_celltype[ct] = cell_type_profiles.nlargest(n_top, ct)[['gene', ct]]

# Create a combined view: heatmap of top genes across all cell types
all_top_genes = list(set([g for ct in top_genes_per_celltype for g in top_genes_per_celltype[ct]['gene'].tolist()]))
available_celltypes = [ct for ct in celltypes if ct in cell_type_profiles.columns]
heatmap_data = cell_type_profiles[cell_type_profiles['gene'].isin(all_top_genes)].set_index('gene')[available_celltypes]

# Normalize per column (z-score) so each cell type has its own scale
heatmap_data_normalized = (heatmap_data - heatmap_data.mean()) / heatmap_data.std()

# Plot heatmap with normalized values
fig, ax = plt.subplots(figsize=(10, max(6, len(all_top_genes) * 0.3)))
sns.heatmap(heatmap_data_normalized, cmap='viridis', ax=ax, cbar_kws={'label': 'Z-score (per cell type)'})
ax.set_title('Top-ranked genes across cell types\n(normalized per cell type)')
ax.set_xlabel('Cell Type')
ax.set_ylabel('Gene')
plt.tight_layout()
plt.show()

In [ ]:
# Bar plot: Top 10 genes for the seed cell type (Macrophage)
fig, ax = plt.subplots(figsize=(8, 5))
if 'Macrophage' in cell_type_profiles.columns:
    top_macro = cell_type_profiles.nlargest(10, 'Macrophage')[['gene', 'Macrophage']]
    ax.barh(top_macro['gene'], top_macro['Macrophage'], color='steelblue')
    ax.set_xlabel('RWR Score')
    ax.set_title('Top 10 upstream regulators in Macrophages')
    ax.invert_yaxis()  # Highest at top
    plt.tight_layout()
    plt.show()
else:
    print("Macrophage not in results - check celltypes list")

### Compare Cell Type Contributions (excluding seed cell type)

Which other cell types contribute to upstream regulation? This reveals which neighboring cells may be sending signals to macrophages:

In [ ]:
# Sum of RWR scores per cell type (excluding seed cell type Macrophage)
seed_celltype = 'Macrophage'
other_celltypes = [ct for ct in celltypes if ct in cell_type_profiles.columns and ct != seed_celltype]
celltype_totals = cell_type_profiles[other_celltypes].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(celltype_totals.index, celltype_totals.values, color='steelblue')
ax.set_xlabel('Total RWR Score')
ax.set_title(f'Upstream contribution from neighboring cell types\n(excluding {seed_celltype} = seed cell type)')
plt.tight_layout()
plt.show()

In [15]:
for key, value in generic_multicell.multiplexes.items():
    print(value)

{'names': ['cell_communication'], 'graph_type': ['11'], 'layers': [                   source             target    weight celltype_source  \
617       Mrc1-Macrophage      Ptprc-NK_cell  2.294999      Macrophage   
713              Mrc1-pDC      Ptprc-NK_cell  2.094999             pDC   
977       Mrc1-Macrophage   Ptprc-T_cell_CD8  1.974999      Macrophage   
1697      Mrc1-Macrophage          Ptprc-pDC  1.839999      Macrophage   
257       Mrc1-Macrophage   Ptprc-Macrophage  1.815000      Macrophage   
1073             Mrc1-pDC   Ptprc-T_cell_CD8  1.774999             pDC   
1793             Mrc1-pDC          Ptprc-pDC  1.639999             pDC   
353              Mrc1-pDC   Ptprc-Macrophage  1.615000             pDC   
857       Mrc1-Macrophage   Ptprc-T_cell_CD4  1.594999      Macrophage   
953              Mrc1-pDC   Ptprc-T_cell_CD4  1.394999             pDC   
17        Mrc1-Macrophage       Ptprc-B_cell  1.149999      Macrophage   
1722         Cd34-NK_cell           Sell-pDC 

In [13]:
print(df)

        Unnamed: 0              target                 source    weight  \
0                0    Pax5::T_cell_CD8    Mbd1_TF::T_cell_CD8  0.000095   
2                2    Pax5::T_cell_CD8   Smad5_TF::T_cell_CD8  0.000092   
1                1    Pax5::T_cell_CD8   Smad1_TF::T_cell_CD8  0.000092   
3                3    Pax5::T_cell_CD8    Mbd2_TF::T_cell_CD8  0.000089   
4                4    Pax5::T_cell_CD8  Zfp128_TF::T_cell_CD8  0.000084   
...            ...                 ...                    ...       ...   
499995      499995   Gata1::T_cell_CD8  Zfp637_TF::T_cell_CD8  0.000001   
499996      499996  Ptpn18::T_cell_CD8  Zfp369_TF::T_cell_CD8  0.000001   
499997      499997  Ptpn18::T_cell_CD8  Zfp110_TF::T_cell_CD8  0.000001   
499998      499998   Cpne2::T_cell_CD8    Mafk_TF::T_cell_CD8  0.000001   
499999      499999   Lyrm7::T_cell_CD8  Tcf7l2_TF::T_cell_CD8  0.000001   

           network_key  
0       T_cell_CD8_grn  
2       T_cell_CD8_grn  
1       T_cell_CD8_grn  

## Exporting multilayer network as .sif file

In [ ]:
import pandas as pd

def export_multicell_to_sif(multicell, filename="multicell_network.sif"):
    edges = []

    # Export multiplex layers
    for key, value in multicell.multiplexes.items():
        df = value["layers"][0]
        tmp = df[["source", "target"]].copy()
        tmp["interaction"] = key
        edges.append(tmp)

    # Export bipartite layers
    for key, value in multicell.bipartites.items():
        df = value["edge_list_df"]
        tmp = df[["col1", "col2"]].copy()
        tmp.columns = ["source", "target"]
        tmp["interaction"] = key
        edges.append(tmp)

    # Concatenate everything
    edges = pd.concat(edges, ignore_index=True)

    # Reorder columns for SIF format
    edges = edges[["source", "interaction", "target"]]

    # Save
    edges.to_csv(filename, sep="\t", header=False, index=False)

    print(f"Network exported to {filename}")

In [ ]:
export_multicell_to_sif(generic_multicell, "multicell_network_ReCoN.sif")